In [1]:
import sys, os

# Add the project root (one level up)
PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

print(PROJECT_ROOT)  # sanity check


c:\Users\anaol\repos\langchain\langchain-AI-summit


In [10]:
import re
from langsmith.schemas import Run, Example
from langsmith import evaluate, aevaluate, wrappers
from openai import OpenAI
from langsmith import Client
from pydantic import BaseModel
from agent.return_agent import agent
from langchain_core.messages import HumanMessage


In [17]:
def prepare_data(run, example):
    # USER QUESTION (from example input)
    user_msg = example.inputs["messages"][0]["content"]

    # REFERENCE ANSWER (from example output)
    ai_messages = [m for m in example.outputs["messages"] if m["type"] == "ai"]
    reference = ai_messages[-1]["content"] if ai_messages else ""

    # MODEL ANSWER (from run)
    output = run.outputs.get("prediction", "")

    return {
        "question": user_msg,
        "reference": reference,
        "answer": output
    }


In [9]:
import uuid

In [16]:
def run_agent(inputs: dict):
    # Extract the user message
    messages = inputs.get("messages", [])
    if not messages:
        raise ValueError("Dataset missing 'messages' list.")

    user_msg = messages[0]["content"]

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Return only final answer
    final_answer = result["messages"][-1].content

    return {"prediction": final_answer}


In [5]:
# Use an LLM-as-a-judge
oai_client = wrappers.wrap_openai(OpenAI())

In [18]:
def relevance(run: Run, example: Example) -> dict:
    """
    Avalia CLAREZA + RELEVÂNCIA da resposta.
    Retorna nota 0..1.
    """
    instructions = """
    Avalie a resposta com base em CLAREZA e RELEVÂNCIA (nota 0 a 1).
    - 1.0 = resposta completa, clara e diretamente relacionada.
    - 0.5 = resposta parcialmente correta.
    - 0.0 = resposta incorreta ou fora do contexto.
    Retorne APENAS a nota, nada mais.
    """

    data = prepare_data(run, example)

    msg = (
        f"Question: {data['question']}\n"
        f"Reference: {data['reference']}\n"
        f"Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    score_str = resp.choices[0].message.content.strip()
    score = float(score_str)

    return {"key": "relevance", "score": score}


In [7]:
client = Client()
client.list_examples(dataset_name="agent-final-output")

<generator object Client.list_examples at 0x000001DE803E6D40>

In [21]:
results = evaluate(
    run_agent,
    data="agent-final-output",
    evaluators=[relevance],
)

View the evaluation results for experiment: 'terrific-son-4' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/cb9bb7e9-467e-48eb-ab7e-1dfa1ff17cac/compare?selectedSessions=1c2f5070-4359-4bb6-baa6-c4947f7c8b97




0it [00:00, ?it/s]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


1it [00:09,  9.71s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


...

5it [01:19, 17.36s/it]

decide_path tool
decision: analyze_seller_reliability
analyze_seller_reliability_node_custom: calling tool directly
Generating final answer...


6it [01:30, 15.46s/it]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


7it [01:44, 14.88s/it]


In [1]:
from langsmith import traceable, wrappers
from openai import OpenAI

# Optionally wrap the OpenAI client to trace all model calls.
oai_client = wrappers.wrap_openai(OpenAI())

# Optionally add the 'traceable' decorator to trace the inputs/outputs of this function.
@traceable
def toxicity_classifier(inputs: dict) -> dict:
    instructions = (
      "Please review the user query below and determine if it contains any form of toxic behavior, "
      "such as insults, threats, or highly negative comments. Respond with 'Toxic' if it does "
      "and 'Not toxic' if it doesn't."
    )
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": inputs["text"]},
    ]
    result = oai_client.chat.completions.create(
        messages=messages, model="gpt-4o-mini", temperature=0
    )
    return {"class": result.choices[0].message.content}

In [2]:
from langsmith import Client
ls_client = Client()

examples = [
  {
    "inputs": {"text": "Shut up, idiot"},
    "outputs": {"label": "Toxic"},
  },
  {
    "inputs": {"text": "You're a wonderful person"},
    "outputs": {"label": "Not toxic"},
  },
  {
    "inputs": {"text": "This is the worst thing ever"},
    "outputs": {"label": "Toxic"},
  },
  {
    "inputs": {"text": "I had a great day today"},
    "outputs": {"label": "Not toxic"},
  },
  {
    "inputs": {"text": "Nobody likes you"},
    "outputs": {"label": "Toxic"},
  },
  {
    "inputs": {"text": "This is unacceptable. I want to speak to the manager."},
    "outputs": {"label": "Not toxic"},
  },
]

dataset = ls_client.create_dataset(dataset_name="Toxic Queries")
ls_client.create_examples(
  dataset_id=dataset.id,
  examples=examples,
)

{'example_ids': ['fe2811fb-95f9-4067-8480-1e6d8375c326',
  '14d9276a-c89e-48e4-9d3a-65ccd5cf6c02',
  '2529fc76-5955-4449-9416-092c1d673451',
  '32d18691-dce7-496a-802b-dcb54e092564',
  '7c0d09ee-3179-472d-a91b-25b50e3b1ed9',
  'b505e8bb-a0d4-4bd6-831d-7c718570469d'],
 'count': 6}

In [3]:
def correct(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    return outputs["class"] == reference_outputs["label"]

In [49]:
# Can equivalently use the 'evaluate' function directly:
from langsmith import evaluate
results = evaluate(
    toxicity_classifier,
    data=dataset.name,
    evaluators=[correct],
    experiment_prefix="gpt-4o-mini, baseline",  # optional, experiment name prefix
    description="Testing the baseline system.",  # optional, experiment description
    max_concurrency=4, # optional, add concurrency
)

View the evaluation results for experiment: 'gpt-4o-mini, baseline-5852482f' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/3a6b2f9e-5c5a-4946-96c3-9107b4e72856/compare?selectedSessions=63c143fe-48bb-4582-983e-bf97f05308f7




0it [00:00, ?it/s]Error running target function: 'coroutine' object has no attribute 'choices'
Traceback (most recent call last):
  File "c:\Program Files\Anaconda\envs\LangChainAcademy\Lib\site-packages\langsmith\evaluation\_runner.py", line 1920, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "c:\Program Files\Anaconda\envs\LangChainAcademy\Lib\site-packages\langsmith\run_helpers.py", line 710, in wrapper
    function_result = run_container["context"].run(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\anaol\AppData\Local\Temp\ipykernel_17752\360263221.py", line 22, in toxicity_classifier
    return {"class": result.choices[0].message.content}
                     ^^^^^^^^^^^^^^
AttributeError: 'coroutine' object has no attribute 'choices'
c:\Program Files\Anaconda\envs\LangChainAcademy\Lib\site-packages\langsmith\evaluation\_runner.py:1927: RuntimeWarning: coroutine 'AsyncCompletions.create' was never awaited
  logger.error(
Error running tar